# llmevalkit - Quick Start Demo

A Python library for evaluating LLM outputs. 15 built-in metrics. Works with or without an API key.

    7 math-based metrics: free, instant, runs offline
    8 LLM-as-judge metrics: uses any LLM provider to evaluate
    Supports: OpenAI, Azure, Anthropic, Groq, Ollama, or no provider at all


In [11]:
!pip install llmevalkit==1.0.3

  Attempting uninstall: llmevalkit
    Found existing installation: llmevalkit 1.0.1
    Uninstalling llmevalkit-1.0.1:
      Successfully uninstalled llmevalkit-1.0.1


## Part A: Math metrics (free, no API key)

In [1]:
from llmevalkit import Evaluator

evaluator = Evaluator(provider="none", preset="math")
result = evaluator.evaluate(
    question="What are the benefits of solar energy?",
    answer="Solar energy is renewable and reduces electricity bills.",
    context="Solar energy is a renewable source that lowers electricity costs."
)
print(result.summary())

  LLMEVAL Evaluation Result
  Question : What are the benefits of solar energy?
  Answer   : Solar energy is renewable and reduces electricity bills.
------------------------------------------------------------
  bleu                 [####................] 0.241
  rouge                [#########...........] 0.464
  token_overlap        [############........] 0.615
  keyword_coverage     [###########.........] 0.571
  answer_length        [####################] 1.000
  readability          [....................] 0.000
------------------------------------------------------------
  Overall Score: 0.482  FAILED


## Individual math metrics

In [2]:
from llmevalkit import BLEUScore, ROUGEScore, TokenOverlap, KeywordCoverage

answer = "Solar panels convert sunlight into electricity efficiently."
context = "Solar panels use photovoltaic cells to convert sunlight into electrical energy."

bleu = BLEUScore()
r = bleu.evaluate(answer=answer, context=context)
print("BLEU: {:.3f}".format(r.score))
print("Precisions: {}".format(r.details["precisions"]))

rouge = ROUGEScore()
r = rouge.evaluate(answer=answer, context=context)
print("ROUGE: {:.3f}".format(r.score))

overlap = TokenOverlap()
r = overlap.evaluate(answer=answer, context=context)
print("Token Overlap: {:.3f}".format(r.score))
print("Common words: {}".format(r.details["common_tokens"]))

kw = KeywordCoverage()
r = kw.evaluate(answer=answer, context=context)
print("Keyword Coverage: {:.3f}".format(r.score))
print("Missing: {}".format(r.details["missing"]))

BLEU: 0.234
Precisions: [0.7143, 0.5, 0.2, 0.0]
ROUGE: 0.501
Token Overlap: 0.533
Common words: ['convert', 'panels', 'solar', 'sunlight']
Keyword Coverage: 0.444
Missing: ['cells', 'electrical', 'energy', 'photovoltaic', 'use']


## Part B: LLM-as-judge metrics (needs API key)

In [6]:
import os
os.environ["GROQ_API_KEY"] = "Your api key"

if os.getenv("GROQ_API_KEY"):
    from llmevalkit import Evaluator
    e = Evaluator(provider="groq", model="llama-3.3-70b-versatile")
    r = e.evaluate(
        question="What are the benefits of solar energy?",
        answer="Solar energy is renewable and reduces electricity bills.",
        context="Solar energy is a renewable source that lowers electricity costs."
    )
    print(r.summary())
else:
    print("Set GROQ_API_KEY to run LLM metrics")

  LLMEVAL Evaluation Result
  Question : What are the benefits of solar energy?
  Answer   : Solar energy is renewable and reduces electricity bills.
------------------------------------------------------------
  faithfulness         [####################] 1.000
  answer_relevance     [###############.....] 0.750
  context_relevance    [##########..........] 0.500
  hallucination        [####################] 1.000
------------------------------------------------------------
  Overall Score: 0.812  PASSED


## Batch evaluation

In [7]:
from llmevalkit import Evaluator

e = Evaluator(provider="none", preset="math")
batch = e.evaluate_batch([
    {"question": "What is Python?", "answer": "Python is a programming language.", "context": "Python is a high-level language."},
    {"question": "What is Python?", "answer": "Yes.", "context": "Python is a high-level language."},
], show_progress=False)

for i, r in enumerate(batch.results):
    print("Case {}: score={:.3f} passed={}".format(i + 1, r.overall_score, r.passed))
print("Pass rate: {:.0%}".format(batch.pass_rate))

Case 1: score=0.605 passed=True
Case 2: score=0.200 passed=False
Pass rate: 50%
